In [ ]:
# ================================================================
# Direct file read from Colab
# Figure texts translated to English
# ================================================================

# If needed, uncomment:
# !pip install -q pandas numpy matplotlib seaborn plotly scipy

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import seaborn as sns
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------
# FILE PATH IN COLAB
# ---------------------------------------------------------------
file_path = "Table1.csv"


def load_csv() -> pd.DataFrame:
    df = pd.read_csv(file_path, low_memory=False)
    print(f"File loaded successfully: {file_path}")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    return df


# ---------------------------------------------------------------
# FIGURE 1 — Age distribution by chronic disease (Top 8)
# ---------------------------------------------------------------
def figure_1_ridgeline_chronic_diseases(df: pd.DataFrame) -> None:
    age_col = "Idade"
    chronic_cols = [c for c in df.columns if "doenças crônicas" in c.lower() and "choice=" in c]

    if age_col not in df.columns:
        raise ValueError(f"Column '{age_col}' was not found in the dataset.")
    if not chronic_cols:
        raise ValueError("No chronic disease checkbox columns were found.")

    df = df.copy()
    df["Age_Num"] = pd.to_numeric(df[age_col], errors="coerce")

    df = df.sort_values(["Record ID", "Repeat Instance"], na_position="first")
    propagate_cols = ["Age_Num"] + chronic_cols
    existing_cols = [c for c in propagate_cols if c in df.columns]
    df[existing_cols] = df.groupby("Record ID")[existing_cols].ffill()

    df_unique = df.drop_duplicates(subset=["Record ID"], keep="last").copy()

    df_melted = df_unique.melt(
        id_vars=["Record ID", "Age_Num"],
        value_vars=chronic_cols,
        var_name="Disease",
        value_name="Status",
    )

    df_diseased = df_melted[df_melted["Status"] == "Checked"].copy()
    df_diseased["Disease_Name"] = (
        df_diseased["Disease"]
        .apply(lambda x: str(x).split("choice=")[-1].replace(")", "").strip())
    )

    useless_terms = ["não se aplica", "nao se aplica", "nenhuma", "nenhum", "outras", "outro"]
    exclusion_pattern = "|".join(useless_terms)
    df_diseased = df_diseased[
        ~df_diseased["Disease_Name"].str.contains(exclusion_pattern, case=False, na=False)
    ]

    df_diseased = df_diseased.dropna(subset=["Age_Num"])

    if df_diseased.empty:
        print("No valid chronic disease records with age information were reported in this dataset.")
        return

    top_diseases = df_diseased["Disease_Name"].value_counts().nlargest(8).index
    df_plot = df_diseased[df_diseased["Disease_Name"].isin(top_diseases)].copy()

    fig = go.Figure()
    colors = px.colors.qualitative.Bold

    for i, disease in enumerate(reversed(top_diseases)):
        age_array = df_plot[df_plot["Disease_Name"] == disease]["Age_Num"]
        current_color = colors[i % len(colors)]
        formatted_name = str(disease)[:35] + "..." if len(str(disease)) > 35 else str(disease)

        fig.add_trace(
            go.Violin(
                x=age_array,
                name=formatted_name,
                side="positive",
                width=2.5,
                points=False,
                line_color="black",
                line_width=1.2,
                fillcolor=current_color,
                opacity=0.85,
                meanline_visible=True,
            )
        )

    fig.update_layout(
        title=dict(
            text="Age Distribution by Chronic Disease (Ridgeline Plot)",
            font=dict(size=20, family="Arial", color="#2c3e50"),
            x=0.05,
        ),
        xaxis=dict(
            title="Patient Age (Years)",
            showgrid=True,
            gridcolor="#f0f0f0",
            zeroline=False,
            title_font=dict(size=14),
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
        ),
        height=700,
        margin=dict(l=280, r=50, t=100, b=80),
        showlegend=False,
        plot_bgcolor="white",
        paper_bgcolor="white",
    )

    fig.show()

    df_table = (
        df_plot.groupby("Disease_Name")["Age_Num"]
        .agg(
            Patient_Count="count",
            Minimum_Age="min",
            Mean_Age="mean",
            Median_Age="median",
            Maximum_Age="max",
        )
        .reset_index()
    )

    df_table["Mean_Age"] = df_table["Mean_Age"].round(1)
    df_table = df_table.sort_values(by="Patient_Count", ascending=False).reset_index(drop=True)

    print("\n" + "=" * 95)
    print("STATISTICAL TABLE: AGE DISTRIBUTION BY CHRONIC DISEASE (Top 8)")
    print("=" * 95)
    print(df_table.to_string(index=False))
    print("=" * 95)

    output_file = "Ridgeline_Age_Chronic_Diseases_Table_EN.csv"
    df_table.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\n[SUCCESS] The base table of the chart was saved as '{output_file}'.\n")


# ---------------------------------------------------------------
# FIGURE 2 — Chronic disease burden from age 45 onward
# ---------------------------------------------------------------
def figure_2_chronic_burden_after_45(df: pd.DataFrame) -> None:
    age_col = "Idade"
    if age_col not in df.columns:
        raise ValueError(f"Column '{age_col}' was not found in the dataset.")

    df = df.copy()
    df["Age_Num"] = pd.to_numeric(df[age_col], errors="coerce")

    df = df.sort_values(["Record ID", "Repeat Instance"], na_position="first")
    df["Age_Num"] = df.groupby("Record ID")["Age_Num"].ffill()

    df_visits = df[df["Repeat Instrument"].notna()].copy()
    df_visits = df_visits.dropna(subset=["Age_Num"])

    df_visits["Age_Group"] = np.where(df_visits["Age_Num"] >= 45, "45 years or older", "Under 45 years")

    text_cols = [c for c in df_visits.columns if df_visits[c].dtype == object]
    df_text = df_visits[text_cols].fillna("")

    regex_htn = r"\b(hipertens[aã]o|has|press[aã]o alta)\b"
    regex_dm = r"\b(diabetes|dm2|dm1|dm)\b"

    cond_htn_text = df_text.apply(
        lambda x: x.astype(str).str.contains(regex_htn, case=False, regex=True)
    ).any(axis=1)

    cond_dm_text = df_text.apply(
        lambda x: x.astype(str).str.contains(regex_dm, case=False, regex=True)
    ).any(axis=1)

    chronic_cols = [c for c in df_visits.columns if "doenças crônicas" in c.lower() and "choice=" in c.lower()]
    cond_htn_struct = pd.Series(False, index=df_visits.index)
    cond_dm_struct = pd.Series(False, index=df_visits.index)

    for c in chronic_cols:
        is_checked = df_visits[c].astype(str).str.lower().str.strip().isin(
            ["checked", "1", "1.0", "sim", "verdadeiro", "true", "yes"]
        )
        if "hipertensão" in c.lower() or "has" in c.lower():
            cond_htn_struct = cond_htn_struct | is_checked
        if "diabetes" in c.lower() or "dm" in c.lower():
            cond_dm_struct = cond_dm_struct | is_checked

    df_visits["Has_Hypertension"] = (cond_htn_text | cond_htn_struct).astype(int)
    df_visits["Has_Diabetes"] = (cond_dm_text | cond_dm_struct).astype(int)

    table_htn = pd.crosstab(df_visits["Age_Group"], df_visits["Has_Hypertension"])
    p_htn = chi2_contingency(table_htn, correction=False)[1]

    table_dm = pd.crosstab(df_visits["Age_Group"], df_visits["Has_Diabetes"])
    p_dm = chi2_contingency(table_dm, correction=False)[1]

    results = []
    groups = ["Under 45 years", "45 years or older"]

    for group in groups:
        df_group = df_visits[df_visits["Age_Group"] == group]
        total_visits = len(df_group)

        if total_visits > 0:
            htn_abs = df_group["Has_Hypertension"].sum()
            dm_abs = df_group["Has_Diabetes"].sum()

            results.append(
                {
                    "Diagnosis": "Hypertension",
                    "Age_Group": group,
                    "Total_Visits": total_visits,
                    "Cases": htn_abs,
                    "Prevalence (%)": round((htn_abs / total_visits) * 100, 2),
                    "P_Value": p_htn,
                }
            )

            results.append(
                {
                    "Diagnosis": "Diabetes",
                    "Age_Group": group,
                    "Total_Visits": total_visits,
                    "Cases": dm_abs,
                    "Prevalence (%)": round((dm_abs / total_visits) * 100, 2),
                    "P_Value": p_dm,
                }
            )

    df_table = pd.DataFrame(results)

    total_overall = len(df_visits)
    total_under45 = len(df_visits[df_visits["Age_Group"] == "Under 45 years"])
    total_45plus = len(df_visits[df_visits["Age_Group"] == "45 years or older"])

    htn_under45 = df_table[
        (df_table["Diagnosis"] == "Hypertension") & (df_table["Age_Group"] == "Under 45 years")
    ]["Prevalence (%)"].values[0]
    htn_45plus = df_table[
        (df_table["Diagnosis"] == "Hypertension") & (df_table["Age_Group"] == "45 years or older")
    ]["Prevalence (%)"].values[0]

    dm_under45 = df_table[
        (df_table["Diagnosis"] == "Diabetes") & (df_table["Age_Group"] == "Under 45 years")
    ]["Prevalence (%)"].values[0]
    dm_45plus = df_table[
        (df_table["Diagnosis"] == "Diabetes") & (df_table["Age_Group"] == "45 years or older")
    ]["Prevalence (%)"].values[0]

    print("\n" + "=" * 85)
    print("STATISTICAL SUMMARY: CHRONIC DISEASE BURDEN AND AGING")
    print("=" * 85)
    print(f"Total Visits Analyzed (with valid age): {total_overall}")
    print(f" -> Under 45 years:      {total_under45}")
    print(f" -> 45 years or older:   {total_45plus}\n")

    print("[1] ARTERIAL HYPERTENSION")
    print(f"    - Prevalence under 45 years: {htn_under45:.2f}%")
    print(f"    - Prevalence 45+ years:      {htn_45plus:.2f}%")
    print(f"    - Statistical increase:      p-value = {p_htn:.4e} (Significant)\n")

    print("[2] DIABETES MELLITUS")
    print(f"    - Prevalence under 45 years: {dm_under45:.2f}%")
    print(f"    - Prevalence 45+ years:      {dm_45plus:.2f}%")
    print(f"    - Statistical increase:      p-value = {p_dm:.4e} (Significant)")
    print("=" * 85 + "\n")

    output_file = "Chronic_Disease_Burden_Age45_Table_EN.csv"
    df_table.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"Table saved as: '{output_file}'.\n")

    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(10, 6))

    colors = ["#4a90e2", "#d55e52"]

    ax = sns.barplot(
        data=df_table,
        x="Diagnosis",
        y="Prevalence (%)",
        hue="Age_Group",
        palette=colors,
        edgecolor="white",
        linewidth=1.5,
    )

    plt.title(
        "Evolution of Chronic Disease Burden From Age 45 Onward",
        fontsize=14,
        pad=20,
        color="#333333",
    )
    plt.ylabel("Prevalence in the Treated Population (%)", fontsize=12, color="#333333")
    plt.xlabel("Diagnosis", fontsize=12, color="#333333")
    plt.ylim(0, max(df_table["Prevalence (%)"]) * 1.3)

    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%", padding=4, fontweight="bold", color="#2c3e50", fontsize=11)

    sns.despine(left=False, bottom=False, top=True, right=True)

    plt.legend(
        title="Age Group",
        title_fontsize="11",
        fontsize="10",
        loc="upper right",
        frameon=True,
    )

    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------------
# RUN EVERYTHING
# ---------------------------------------------------------------
def main():
    df = load_csv()
    figure_1_ridgeline_chronic_diseases(df)
    figure_2_chronic_burden_after_45(df)


main()